In [ ]:
# ============================================================================
# CELL 1: Setup Google Colab Environment
# Run this cell FIRST
# ============================================================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install -q openai tqdm

print("✓ Setup complete!")

# ============================================================================
# CELL 2: Import Libraries and Configuration
# ============================================================================

import pandas as pd
import numpy as np
import json
import time
import os
from datetime import datetime
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from openai import OpenAI
import openai

# Configuration
INPUT_FILE = "/content/drive/MyDrive/PropInsight/preprocess/forum/singapore_property_forum_posts_sgexpats_processed/forum_enriched.csv"
OUTPUT_FILE = "/content/drive/MyDrive/PropInsight/labeled/singapore_property_forum_posts_sgexpats_processed/forum_labeled.csv"


# Checkpoint paths (FIX)
CHECKPOINT_FILE = "/content/drive/MyDrive/PropInsight/labeled/checkpoint_sgexpats/singapore_property_forum_posts_sgexpats_processed/labeling_checkpoint.csv"

# Make the *directory*, not the file path
CHECKPOINT_DIR = os.path.dirname(CHECKPOINT_FILE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# If a directory was mistakenly created at the file path, auto-repair it
if os.path.isdir(CHECKPOINT_FILE):
    # Move that mistaken directory aside so we can use the file path properly
    repaired_dir = CHECKPOINT_FILE + "_backup_dir"
    print(f"⚠ Found a directory at checkpoint file path. Moving it to: {repaired_dir}")
    os.rename(CHECKPOINT_FILE, repaired_dir)


# Make the *directory*, not the file path
OUTPUT_DIR = os.path.dirname(OUTPUT_FILE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If a directory was mistakenly created at the file path, auto-repair it
if os.path.isdir(OUTPUT_FILE):
    # Move that mistaken directory aside so we can use the file path properly
    repaired_dir = OUTPUT_FILE + "_backup_dir"
    print(f"⚠ Found a directory at checkpoint file path. Moving it to: {repaired_dir}")
    os.rename(OUTPUT_FILE, repaired_dir)





# OpenAI Settings
OPENAI_MODEL = "gpt-4o"  # Cost-effective and fast
MAX_RETRIES = 3
RETRY_DELAY = 2
BATCH_SIZE = 10  # Save checkpoint every 10 posts
RATE_LIMIT_DELAY = 1  # 1 second between API calls

print("✓ Configuration loaded")

# ============================================================================
# CELL 3: Enter Your OpenAI API Key
# Get your key from: https://platform.openai.com/api-keys
# ============================================================================

from google.colab import userdata

try:
    # Get API key from Colab secrets
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

    # Initialize OpenAI client
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("✓ OpenAI API key loaded from Colab secrets")
    print("✓ OpenAI client initialized")

    # Test the connection
    test_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "test"}],
        max_tokens=5
    )
    print("✓ API connection successful!")
    print(f"✓ Using model: {OPENAI_MODEL}")

except Exception as e:
    print("✗" * 70)
    print("ERROR: Could not load OpenAI API key from Colab secrets")
    print("✗" * 70)
    print(f"\nError details: {str(e)}")
    print("\nTo fix this:")
    print("1. Click the 🔑 key icon in the left sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: OPENAI_API_KEY")
    print("4. Value: <paste your API key>")
    print("5. Toggle 'Notebook access' ON")
    print("6. Re-run this cell")
    print("\nGet your API key from: https://platform.openai.com/api-keys")
    raise


# ============================================================================
# CELL 4: Define Helper Functions
# ============================================================================

def extract_source_from_url(url: str) -> str:
    """Extract forum source from URL."""
    if pd.isna(url):
        return "unknown"
    url_lower = url.lower()
    if "hardwarezone" in url_lower or "hwz" in url_lower:
        return "HWZ"
    elif "singaporeexpats" in url_lower or "sgexpats" in url_lower:
        return "SGExpats"
    return "unknown"


def create_llm_prompt(text: str, has_singlish: bool, policy_flags: Dict) -> str:
    """Create prompt for OpenAI to analyze the forum post."""

    policy_info = []
    for policy, flag in policy_flags.items():
        if flag:
            policy_name = policy.replace('flag_', '')
            policy_info.append(policy_name)

    policy_context = ", ".join(policy_info) if policy_info else "None"

    prompt = f"""Analyze this Singapore property forum post and extract sentiment and context.

POST TEXT:
{text[:2000]}

EXISTING CONTEXT:
- Singlish detected: {has_singlish}
- Policies mentioned: {policy_context}

Extract these fields in JSON format:

1. overall_sentiment: Overall market optimism ("positive", "neutral", "negative")
2. price_sentiment: Price direction perception ("rising", "neutral", "falling")
3. policy_sentiment: Attitude toward government policies ("positive", "neutral", "negative")
4. affordability_sentiment: Affordability perception ("positive", "neutral", "negative")
5. location: Singapore locations mentioned (e.g., "Woodlands, D19" or "none")
6. policy_mentioned: Policy terms found (e.g., "ABSD, TDSR" or "none")
7. cultural_context: Singaporean cultural meaning (e.g., "kiasu urgency" or "none")
8. emotion: Dominant emotion ("joy", "anger", "fear", "trust", "anticipation", "surprise", "sadness", "disgust", "neutral")

IMPORTANT:
- Be specific with locations (neighborhood names, districts)
- Policy sentiment = attitude toward government intervention
- Affordability sentiment = whether poster thinks it's affordable
- Cultural context = uniquely Singaporean perspectives (kiasu, pragmatic, FOMO, etc.)

Return ONLY valid JSON:
{{
  "overall_sentiment": "",
  "price_sentiment": "",
  "policy_sentiment": "",
  "affordability_sentiment": "",
  "location": "",
  "policy_mentioned": "",
  "cultural_context": "",
  "emotion": ""
}}"""

    return prompt


def call_openai_api(client: OpenAI, prompt: str) -> Dict:
    """Call OpenAI API with retry logic."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert in analyzing Singapore property forum discussions. You understand Singlish, local context, and market sentiment. Respond with valid JSON only."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=500,
                response_format={"type": "json_object"}
            )

            result = json.loads(response.choices[0].message.content)
            return result

        except openai.RateLimitError:
            wait_time = RETRY_DELAY * (attempt + 1)
            print(f"⚠ Rate limit. Waiting {wait_time}s...")
            time.sleep(wait_time)

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                print(f"⚠ Error (attempt {attempt+1}): {str(e)}")
                time.sleep(RETRY_DELAY)
            else:
                print(f"✗ Failed after {MAX_RETRIES} attempts")

    # Return defaults if all attempts fail
    return {
        "overall_sentiment": "neutral",
        "price_sentiment": "neutral",
        "policy_sentiment": "neutral",
        "affordability_sentiment": "neutral",
        "location": "none",
        "policy_mentioned": "none",
        "cultural_context": "none",
        "emotion": "neutral"
    }


def process_single_post(row: pd.Series, client: OpenAI) -> Dict:
    """Process one forum post and return all labels."""

    # Extract existing data
    text = str(row.get('clean_text', ''))
    has_singlish = bool(row.get('has_singlish', False))

    # Get policy flags
    policy_flags = {
        'flag_ABSD': row.get('flag_ABSD', False),
        'flag_BSD': row.get('flag_BSD', False),
        'flag_SSD': row.get('flag_SSD', False),
        'flag_TDSR': row.get('flag_TDSR', False),
        'flag_LTV': row.get('flag_LTV', False),
        'flag_MSR': row.get('flag_MSR', False),
        'flag_HLE': row.get('flag_HLE', False)
    }

    # Call OpenAI API
    prompt = create_llm_prompt(text, has_singlish, policy_flags)
    llm_result = call_openai_api(client, prompt)

    # Combine results with simple derived fields
    labels = {
        'overall_sentiment': llm_result.get('overall_sentiment', 'neutral'),
        'price_sentiment': llm_result.get('price_sentiment', 'neutral'),
        'policy_sentiment': llm_result.get('policy_sentiment', 'neutral'),
        'affordability_sentiment': llm_result.get('affordability_sentiment', 'neutral'),
        'location': llm_result.get('location', 'none'),
        'policy_mentioned': llm_result.get('policy_mentioned', 'none'),
        'singlish_detected': has_singlish,
        'cultural_context': llm_result.get('cultural_context', 'none'),
        'emotion': llm_result.get('emotion', 'neutral'),
        'source': extract_source_from_url(row.get('thread_url', ''))
    }

    return labels

print("✓ Helper functions defined")

# ============================================================================
# CELL 5: Load Data and Initialize
# ============================================================================

print("Loading forum data...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"✓ Loaded {len(df)} forum posts")
    print(f"  Columns: {len(df.columns)}")
    print(f"  First few columns: {list(df.columns)[:5]}")
except FileNotFoundError:
    print(f"✗ File not found: {INPUT_FILE}")
    print("Please check the path and try again.")
    raise
except Exception as e:
    print(f"✗ Error: {str(e)}")
    raise

# Check for checkpoint
print("\nChecking for existing checkpoint...")
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    try:
        checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
        if 'overall_sentiment' in checkpoint_df.columns:
            labeled_count = checkpoint_df['overall_sentiment'].notna().sum()
            if labeled_count > 0:
                print(f"✓ Found checkpoint with {labeled_count} labeled posts")
                df = checkpoint_df
                start_idx = labeled_count
                print(f"  Resuming from row {start_idx}")
    except Exception as e:
        print(f"⚠ Checkpoint exists but couldn't load: {str(e)}")
        print("  Starting fresh...")
else:
    print("  No checkpoint found")

# Initialize label columns if needed
label_columns = [
    'overall_sentiment', 'price_sentiment', 'policy_sentiment',
    'affordability_sentiment', 'location', 'policy_mentioned',
    'singlish_detected', 'cultural_context', 'emotion', 'source'
]

for col in label_columns:
    if col not in df.columns:
        df[col] = None

print(f"\n✓ Ready to process {len(df) - start_idx} posts")
print(f"  Model: {OPENAI_MODEL}")
print(f"  Checkpoint frequency: every {BATCH_SIZE} posts")

# ============================================================================
# CELL 6: Process All Posts (Main Loop)
# ============================================================================

print("=" * 70)
print("Starting labeling process...")
print("=" * 70)
print()

total_posts = len(df)
posts_to_process = total_posts - start_idx

# Progress bar
pbar = tqdm(total=posts_to_process, desc="Labeling", unit="posts")

processed_count = 0
errors = []

try:
    for idx in range(start_idx, total_posts):

        # Skip if already labeled
        if pd.notna(df.at[idx, 'overall_sentiment']):
            pbar.update(1)
            continue

        try:
            # Process the post
            labels = process_single_post(df.iloc[idx], client)

            # Update dataframe
            for key, value in labels.items():
                df.at[idx, key] = value

            processed_count += 1
            pbar.update(1)

            # Save checkpoint
            if processed_count % BATCH_SIZE == 0:
                df.to_csv(CHECKPOINT_FILE, index=False)
                pbar.set_postfix({
                    "saved": f"{processed_count} posts",
                    "errors": len(errors)
                })

            # Rate limiting
            time.sleep(RATE_LIMIT_DELAY)

        except KeyboardInterrupt:
            print("\n\n⚠ Interrupted by user")
            print("Saving checkpoint...")
            df.to_csv(CHECKPOINT_FILE, index=False)
            raise

        except Exception as e:
            error_msg = f"Row {idx}: {str(e)}"
            errors.append(error_msg)
            print(f"\n⚠ {error_msg}")
            continue

    pbar.close()

except KeyboardInterrupt:
    print("\nProcess stopped by user. Checkpoint saved.")
    pbar.close()

print()
print("=" * 70)
print("Processing complete!")
print("=" * 70)
print(f"  Total processed: {processed_count}")
print(f"  Errors: {len(errors)}")

if errors:
    print("\nError summary:")
    for err in errors[:5]:  # Show first 5 errors
        print(f"  - {err}")
    if len(errors) > 5:
        print(f"  ... and {len(errors)-5} more")

# ============================================================================
# CELL 7: Save Final Output and View Statistics
# ============================================================================

print("\nSaving final labeled dataset...")
try:
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"✓ Saved to: {OUTPUT_FILE}")
    print(f"  Total rows: {len(df)}")
except Exception as e:
    print(f"✗ Error saving: {str(e)}")
    print(f"  Checkpoint available at: {CHECKPOINT_FILE}")

print("\n" + "=" * 70)
print("SUMMARY STATISTICS")
print("=" * 70)

print("\n📊 Overall Sentiment:")
print(df['overall_sentiment'].value_counts())

print("\n📈 Price Sentiment:")
print(df['price_sentiment'].value_counts())

print("\n🏛️ Policy Sentiment:")
print(df['policy_sentiment'].value_counts())

print("\n💰 Affordability Sentiment:")
print(df['affordability_sentiment'].value_counts())

print("\n😊 Emotion Distribution:")
print(df['emotion'].value_counts())

print("\n🗣️ Singlish Detection:")
singlish_count = df['singlish_detected'].sum()
print(f"  Posts with Singlish: {singlish_count} ({singlish_count/len(df)*100:.1f}%)")
print(f"  Posts without: {len(df)-singlish_count} ({(len(df)-singlish_count)/len(df)*100:.1f}%)")

print("\n📍 Top Locations (excluding 'none'):")
locations_df = df[df['location'] != 'none']['location'].str.split(',').explode().str.strip()
top_locations = locations_df.value_counts().head(10)
print(top_locations)

print("\n📋 Policies Mentioned (excluding 'none'):")
policies_df = df[df['policy_mentioned'] != 'none']['policy_mentioned'].str.split(',').explode().str.strip()
top_policies = policies_df.value_counts()
print(top_policies)

print("\n🏠 Source Distribution:")
print(df['source'].value_counts())

print("\n🎭 Cultural Context Examples:")
cultural_examples = df[df['cultural_context'] != 'none']['cultural_context'].value_counts().head(10)
print(cultural_examples)

print("\n" + "=" * 70)
print("✓ All done! 🎉")
print("=" * 70)

# ============================================================================
# CELL 8: (Optional) View Sample Labeled Posts
# ============================================================================

print("Sample labeled posts:\n")
print("=" * 70)

sample_df = df[df['overall_sentiment'].notna()].sample(min(5, len(df)))

for idx, row in sample_df.iterrows():
    print(f"\n📝 POST #{idx}")
    print(f"Text (excerpt): {str(row['clean_text'])[:200]}...")
    print(f"\nLabels:")
    print(f"  - Overall Sentiment: {row['overall_sentiment']}")
    print(f"  - Price Sentiment: {row['price_sentiment']}")
    print(f"  - Policy Sentiment: {row['policy_sentiment']}")
    print(f"  - Affordability: {row['affordability_sentiment']}")
    print(f"  - Location: {row['location']}")
    print(f"  - Policy Mentioned: {row['policy_mentioned']}")
    print(f"  - Emotion: {row['emotion']}")
    print(f"  - Cultural Context: {row['cultural_context']}")
    print(f"  - Singlish: {'Yes' if row['singlish_detected'] else 'No'}")
    print(f"  - Source: {row['source']}")
    print("-" * 70)

print("\n✓ Done viewing samples!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Setup complete!
✓ Configuration loaded
✓ OpenAI API key loaded from Colab secrets
✓ OpenAI client initialized
✓ API connection successful!
✓ Using model: gpt-4o
✓ Helper functions defined
Loading forum data...
✓ Loaded 355 forum posts
  Columns: 49
  First few columns: ['thread_url', 'post_text', 'date', 'raw_text', 'clean_text']

Checking for existing checkpoint...
✓ Found checkpoint with 350 labeled posts
  Resuming from row 350

✓ Ready to process 5 posts
  Model: gpt-4o
  Checkpoint frequency: every 10 posts
Starting labeling process...



Labeling:   0%|          | 0/5 [00:00<?, ?posts/s]


Processing complete!
  Total processed: 5
  Errors: 0

Saving final labeled dataset...
✓ Saved to: /content/drive/MyDrive/PropInsight/labeled/singapore_property_forum_posts_sgexpats_processed/forum_labeled.csv
  Total rows: 355

SUMMARY STATISTICS

📊 Overall Sentiment:
overall_sentiment
negative    196
neutral     145
positive     14
Name: count, dtype: int64

📈 Price Sentiment:
price_sentiment
neutral    163
rising     128
falling     62
peaking      2
Name: count, dtype: int64

🏛️ Policy Sentiment:
policy_sentiment
neutral     291
negative     63
positive      1
Name: count, dtype: int64

💰 Affordability Sentiment:
affordability_sentiment
negative    192
neutral     140
positive     23
Name: count, dtype: int64

😊 Emotion Distribution:
emotion
disgust           91
neutral           84
frustration       49
anticipation      46
fear              22
trust             18
anger             13
surprise          11
concern            9
sadness            7
joy                3
disappointme